   
# Lab: HIPAA-Compliant Patient Analytics
## Your Step-by-Step Guide

This notebook walks you through building a **HIPAA-compliant data governance layer** on Databricks using Unity Catalog. Every step tells you **exactly which cell to run**, **what output to expect**, and **what to do if something goes wrong**.

---

### 🧪 This Is a Hands-On Lab

This notebook contains **6 coding exercises** where key parts of the code have been removed. Look for `<FILL_IN>` placeholders — replace them with the correct values to make each cell run successfully.

| Exercise | Section | Concept Tested |
|---|---|---|
| Exercise 1 | Section 2 | Apply PHI classification tags |
| Exercise 2 | Section 2 | Query information_schema for compliance reporting |
| Exercise 3 | Section 3 | Create a dynamic data masking function |
| Exercise 4 | Section 3 | Apply column masks to a table |
| Exercise 5 | Section 4 | Create a row-level security filter |
| Exercise 6 | Section 5 | Query system audit logs |

> **💡 Tip:** Each `<FILL_IN>` has an inline comment hint to guide you. If you get truly stuck, scroll to the **Answer Key** at the very end of this notebook.

---

### What You Will Learn

By the end of this lab, you will be able to:
1. **Configure Unity Catalog** for healthcare data governance
2. **Classify sensitive data** using Unity Catalog tags
3. **Implement Dynamic Data Masking** to protect PHI (Protected Health Information)
4. **Apply Row-Level Security** to restrict data access by department/region
5. **Monitor data access** using system audit logs for HIPAA compliance reporting

### HIPAA Quick Reference
| HIPAA Requirement | Databricks Feature | Where in This Lab |
|---|---|---|
| Access Controls (§164.312(a)) | Unity Catalog RBAC | Section 2–4 |
| Minimum Necessary (§164.502(b)) | Column Masking + Row Filters | Section 3–4 |
| Audit Controls (§164.312(b)) | System Audit Logs | Section 5 |
| Data Classification | UC Tags + Comments | Section 2 |
| Encryption at Rest | Managed by AWS/Databricks | Explained in Section 1 |
| Encryption in Transit | TLS 1.2+ (built-in) | Explained in Section 1 |

---

### How to Run a Cell

1. Click anywhere inside a code cell to select it (a blue border appears)
2. Click the **▶ Run** button in the top-right corner of the cell, **OR** press **Shift + Enter** on your keyboard
3. Wait for the cell to finish — you will see output appear below it and any spinner will stop
4. **Do not skip cells** — run them in the order listed in these instructions

### How to Find a Cell

Every code cell in this notebook has a **title** displayed above it (in bold text at the top of the cell). These instructions refer to cells by their exact title. For example:

> **Run the cell titled "Verify Unity Catalog and Tables"**

…means scroll through the notebook until you find a cell with that exact title, then click **▶ Run**.

---

### Prerequisites — Before You Start

1. **Databricks Workspace Access** — You must be logged into a Databricks workspace with Unity Catalog enabled
2. **Compute Available** — Serverless compute will be used automatically. No cluster setup is needed
3. **Account-Level Group Created** — You must create an `account_managers` group at the account level before running Section 1 (detailed instructions are provided below)
4. **Run Sections in Order** — This notebook has 6 sections. Run them top-to-bottom. Do not jump ahead

---
# Section 1: Unity Catalog Setup & Healthcare Data Architecture

**Goal:** Set up the demo environment by running two setup notebooks and verifying that all clinical data tables are created.

---

### Background

**What is Unity Catalog?** Unity Catalog is Databricks’ unified governance solution that provides centralized access control, fine-grained permissions (column masking, row filters), data lineage tracking, and audit logging for compliance.

**Three-Level Namespace:** All tables in this lab live inside a three-level structure:
```
Catalog (hipaa_demo)
  └── Schema (clinical_data)
        ├── Table (patients)        ← 10,000 synthetic patient records
        ├── Table (encounters)      ← clinical encounters
        ├── Table (diagnoses)       ← diagnosis codes
        ├── Table (lab_results)     ← lab test results
        └── Table (medications)     ← prescribed medications
```

Follow the steps below in order. Each step instruction appears **directly above** the cell you need to act on.

---

### Step 1.1 - Production Architecture: IAM Roles & S3 Storage

> **This cell is for reading only — no action needed.** The steps below explain what a production HIPAA-compliant deployment looks like. They cannot be executed on a free trial. Scroll past this cell when you are done reading.

### 1. AWS IAM Configuration (Production)

In production, you need a dedicated IAM role for the Unity Catalog metastore:

```json
// IAM Trust Policy for Unity Catalog Metastore
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Principal": {
        "AWS": "arn:aws:iam::DATABRICKS_ACCOUNT_ID:role/unity-catalog-prod"
      },
      "Action": "sts:AssumeRole",
      "Condition": {
        "StringEquals": {
          "sts:ExternalId": "YOUR_EXTERNAL_ID"
        }
      }
    }
  ]
}
```

```json
// S3 Bucket Policy for HIPAA Data
{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "s3:GetObject",
        "s3:PutObject",
        "s3:DeleteObject",
        "s3:ListBucket",
        "s3:GetBucketLocation"
      ],
      "Resource": [
        "arn:aws:s3:::hipaa-clinical-data-prod",
        "arn:aws:s3:::hipaa-clinical-data-prod/*"
      ]
    }
  ]
}
```

### 2. S3 Bucket Requirements for HIPAA
| Setting | Required Value | Why |
|---------|---------------|-----|
| Server-Side Encryption | AES-256 or AWS KMS | HIPAA §164.312(a)(2)(iv) |
| Versioning | Enabled | Data recovery & audit trail |
| Access Logging | Enabled → audit bucket | HIPAA §164.312(b) |
| Public Access | **Block ALL** | Prevent data exposure |
| VPC Endpoint | Recommended | Keep traffic on private network |
| Cross-Region Replication | Optional | Disaster recovery |

### 3. What We CAN Do on This Trial Workspace
The trial workspace comes with:
* **Unity Catalog** pre-configured with a default metastore
* **Managed storage** (Databricks-managed S3 bucket with encryption)
* **Catalog creation** privileges
* **Column masking & row filters**
* **Tags and data classification**
* **Audit logs** (system tables)

> **➡️ When you are done reading, scroll down to continue with Step 1.2.**

### Step 1.2 - Reference: Create the `account_managers` Group

1. Open a **new browser tab** and go to the [Databricks Account Console](https://accounts.cloud.databricks.com) and log in as an **account admin**
2. In the left sidebar, click **User management**
3. Select the **Groups** tab
4. Click **Add group**
5. Enter the group name: **`account_managers`** (must match exactly)
6. Click **Save**
7. In the sidebar, click **Workspaces**
8. Click your workspace name (workspace)
9. On the Permissions tab, click **Add permissions**
10. Search for and select the group **account_managers**, assign the permission level **Admin**, and then click **Save**
11. Return to this notebook and continue to the next cell

**Important Note**: If you see the error `PRINCIPAL_DOES_NOT_EXIST`, double-check the group name spelling and confirm it was created at the **account level**, not the workspace level

### Step 1.3 - Run the Groups and Users Setup

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Output from the setup notebook confirming that security groups (`phi_authorized`, `research_analysts`, `billing_team`, etc.) were created or already exist
3. **If you see an error about `account_managers`:** Go back to Step 1.2 above and verify the group was created correctly at the **account level**

In [0]:
%run "./00a - Setup Groups and Users"

### Step 1.4 - Run the Synthetic Patient Data Setup

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** Output confirming that the `hipaa_demo` catalog, `clinical_data` schema, and five tables were created with synthetic data
3. **This cell may take 1–2 minutes** to complete because it generates 10,000 patient records and related clinical data. Wait for it to finish before moving on

In [0]:
%run "./00b - Setup Synthetic Patient Data"

### Step 1.5 - Verify Unity Catalog and Tables

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A table with 5 rows showing each table name and its row count:
   ```
   table_name     | row_count
   diagnoses      | ~30,000
   encounters     | ~30,000
   lab_results    | ~40,000
   medications    | ~25,000
   patients       | 10,000
   ```
3. **If any row count is 0 or the query fails:** Go back and re-run Steps 1.3 and 1.4 in order

In [0]:
%sql
-- =============================================================================
-- Verify our catalog and schema are set up correctly
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- Show all tables and their row counts
SELECT 
  'patients' AS table_name, COUNT(*) AS row_count FROM patients
UNION ALL
SELECT 'encounters', COUNT(*) FROM encounters
UNION ALL
SELECT 'diagnoses', COUNT(*) FROM diagnoses
UNION ALL
SELECT 'lab_results', COUNT(*) FROM lab_results
UNION ALL
SELECT 'medications', COUNT(*) FROM medications
ORDER BY table_name

### Step 1.6 - Preview Patient PHI Data (Before Masking)

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A table showing 10 patient records with **all PHI fields fully visible** — names, SSNs, dates of birth, phone numbers, emails, and addresses
3. **Take a moment to look at the data.** This is the sensitive information we need to protect. By the end of Section 3, these same columns will be automatically masked

In [0]:
%sql
-- =============================================================================
-- Look at the patient data BEFORE we apply any masking
-- Notice: ALL PHI fields are fully visible (SSN, name, DOB, address, etc.)
-- This is what we need to PROTECT!
-- =============================================================================

SELECT 
  patient_id,
  first_name,
  last_name,
  ssn,
  date_of_birth,
  phone_number,
  email,
  address,
  city,
  state,
  insurance_type,
  region,
  primary_department
FROM hipaa_demo.clinical_data.patients
LIMIT 10

> ✅ **Checkpoint:** You have run the setup notebooks and verified that all 5 clinical data tables exist with data. You have also seen the raw, unmasked patient data. **Scroll down to continue to Section 2.**

---
# Section 2: Data Classification with Unity Catalog Tags

**Goal:** Label every column that contains sensitive information (PHI) using Unity Catalog tags. This lets you generate compliance reports showing exactly where sensitive data lives.

---

### Background

**Why classify healthcare data?** HIPAA requires organizations to identify and categorize Protected Health Information (PHI). Unity Catalog **tags** let you:
* Label columns containing PHI, PII, or sensitive data
* Create policies based on classification
* Generate compliance reports showing where sensitive data lives

**The 18 HIPAA identifiers** include: Names, Social Security numbers, dates related to an individual, phone numbers, email addresses, geographic data (smaller than state), medical record numbers, health plan IDs, and more.

Follow the steps below in order.

---

### Step 2.1 - 🧪 Exercise 1: Apply PHI Tags to the Patients Table

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A single row of output:
   ```
   status
   ✅ Tags applied successfully to patients table
   ```
4. **What the cell does:** Applies tags to the `patients` table at two levels:
   * **Table-level tags:** `data_classification = PHI`, `hipaa_covered = true`, `retention_policy = 7_years`
   * **Column-level tags:** Each PHI column (like `ssn`, `first_name`, `date_of_birth`) is tagged with its sensitivity level (`critical`, `high`) and PHI category
   * Non-PHI columns (like `gender`, `blood_type`) are tagged as `sensitivity = low` and `analytics_safe = true`

**Hints:**
* HIPAA defines 18 types of Protected Health Information — the table-level classification should reflect this
* The sensitivity levels used in this lab are: `critical` (for identifiers like SSN), `high` (for names, dates, addresses), `medium`, and `low`
* The PHI categories match the data type: names → `'name'`, social security numbers → think about what abbreviation is commonly used

In [0]:
%sql
-- =============================================================================
-- EXERCISE 1: Apply sensitivity tags to columns containing PHI
-- Tags enable automated governance and compliance reporting
--
-- Replace each <FILL_IN> with the correct value to make this cell run.
-- =============================================================================

-- First, set table-level tags
ALTER TABLE hipaa_demo.clinical_data.patients 
  SET TAGS ('data_classification' = <FILL_IN>,             -- What type of data does this table contain? (Hint: Protected Health ___)
            'hipaa_covered' = 'true',
            'retention_policy' = '7_years',
            'data_owner' = 'clinical_ops');

-- Tag individual PHI columns
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET TAGS ('phi_category' = 'name', 'sensitivity' = <FILL_IN>);   -- Names are sensitive — what level? (critical, high, medium, or low)

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET TAGS ('phi_category' = 'name', 'sensitivity' = <FILL_IN>);    -- Same sensitivity as first_name

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET TAGS ('phi_category' = <FILL_IN>, 'sensitivity' = 'critical');       -- What PHI category is a Social Security Number?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET TAGS ('phi_category' = 'date', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET TAGS ('phi_category' = <FILL_IN>, 'sensitivity' = 'high');       -- Addresses fall under which PHI category? (Hint: it's about location)

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET TAGS ('phi_category' = 'phone', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET TAGS ('phi_category' = 'email', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET TAGS ('phi_category' = 'health_plan_id', 'sensitivity' = 'high');

-- Tag non-PHI columns as safe for analytics
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN gender SET TAGS ('sensitivity' = 'low', 'analytics_safe' = <FILL_IN>);      -- Should gender be available for analytics? ('true' or 'false')

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN race SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN blood_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

SELECT '✅ Tags applied successfully to patients table' AS status

### Step 2.2 - Apply Tags to the Other Clinical Tables

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:**
   ```
   status
   ✅ Tags applied to all clinical tables
   ```
3. **What just happened:** Tags were applied to the `encounters`, `diagnoses`, `lab_results`, and `medications` tables. For example, `icd10_code` in the diagnoses table was tagged as `data_type = medical_code` with `sensitivity = medium`

In [0]:
%sql
-- =============================================================================
-- Tag remaining clinical tables
-- =============================================================================

-- Encounters table
ALTER TABLE hipaa_demo.clinical_data.encounters 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

-- Diagnoses table  
ALTER TABLE hipaa_demo.clinical_data.diagnoses 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.diagnoses 
  ALTER COLUMN icd10_code SET TAGS ('data_type' = 'medical_code', 'sensitivity' = 'medium');

-- Lab Results table
ALTER TABLE hipaa_demo.clinical_data.lab_results 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.lab_results 
  ALTER COLUMN result_value SET TAGS ('data_type' = 'clinical_measurement', 'sensitivity' = 'medium');

-- Medications table
ALTER TABLE hipaa_demo.clinical_data.medications 
  SET TAGS ('data_classification' = 'clinical', 'hipaa_covered' = 'true');

ALTER TABLE hipaa_demo.clinical_data.medications 
  ALTER COLUMN medication_name SET TAGS ('data_type' = 'prescription', 'sensitivity' = 'medium');

SELECT '✅ Tags applied to all clinical tables' AS status

### Step 2.3 - 🧪 Exercise 2: Generate a Compliance Report

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A table listing every tagged column across all clinical tables, with columns:
   * `table_name` — which table the column belongs to
   * `column_name` — the name of the tagged column
   * `tag_name` — the tag that was applied (e.g., `phi_category`, `sensitivity`)
   * `tag_value` — the value of the tag (e.g., `ssn`, `critical`)
4. **Take a moment to scroll through the results.** This is exactly the kind of report a compliance officer would use to verify that all sensitive data has been identified

**Hints:**
* You are joining three `information_schema` views: `tables`, `columns`, and `column_tags`
* The JOIN between `tables` and `columns` already matches on `table_schema` — what other field needs to match?
* The LEFT JOIN to `column_tags` already matches on `schema_name` and `table_name` — what column-level field completes the match?
* The schema name is the one you saw in Section 1: `clinical_data`

In [0]:
%sql
-- =============================================================================
-- EXERCISE 2: Generate a compliance report — Which columns contain PHI?
-- This query uses the information_schema to find all tagged columns
--
-- Replace each <FILL_IN> with the correct value to make this cell run.
-- =============================================================================

SELECT 
  t.table_name,
  c.column_name,
  c.data_type,
  ct.tag_name,
  ct.tag_value
FROM hipaa_demo.information_schema.tables t
JOIN hipaa_demo.information_schema.columns c 
  ON t.table_schema = c.table_schema AND t.table_name = <FILL_IN>      -- Join tables to columns on which field?
LEFT JOIN hipaa_demo.information_schema.column_tags ct
  ON c.table_schema = ct.schema_name 
  AND c.table_name = ct.table_name 
  AND c.column_name = <FILL_IN>                                        -- Match tags to columns on which field?
WHERE t.table_schema = <FILL_IN>                                       -- Which schema holds our clinical tables? (a string literal)
  AND ct.tag_name IS NOT NULL
ORDER BY t.table_name, c.ordinal_position, ct.tag_name

> ✅ **Checkpoint:** All 5 clinical tables are now tagged with sensitivity classifications. You can see a complete inventory of where PHI lives in your database. **Scroll down to continue to Section 3.**

---
# Section 3: Dynamic Data Masking (Column-Level Security)

**Goal:** Create masking functions that automatically hide or partially reveal PHI based on who is running the query. Then apply those masks to the patients table and see them in action.

---

### Background

**How does column masking work?** When you query a table, Unity Catalog checks if the column has a mask applied. If it does, it runs a SQL function that decides what to show you based on your group membership.

```
You run a query → Unity Catalog checks your groups → Masking function runs → You see masked or full data
```

**Our access model** defines four levels of access:

| Group | What They See | Use Case |
|-------|--------------|----------|
| `phi_authorized` | Full PHI — nothing masked | Clinical staff, treating physicians |
| `research_analysts` | Partial masking (last 4 of SSN, year of birth only, initials) | Research team |
| `billing_team` | Names + insurance info only | Revenue cycle |
| Default (no group) | Fully masked — `[REDACTED]` or `***` | Data scientists, general analysts |

Follow the steps below in order.

---

### Step 3.1 - 🧪 Exercise 3: Create the SSN Masking Function

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** The cell completes with no error (a small green checkmark or "OK" status). No visible output table is expected — it just creates the function
4. **What it does:** Creates a function called `mask_ssn` that:
   * Shows the full SSN if you are in the `phi_authorized` group
   * Shows `***-**-6789` (last 4 digits only) if you are in `research_analysts`
   * Shows `***-**-****` (fully hidden) for everyone else

**Hints:**
* The `IS_MEMBER()` function takes a group name as a string — look at the access model table in the Section 3 header above
* The first group that should see full SSNs has "authorized" in its name
* The second group that gets partial access is the research team
* The default (ELSE) case should return a string that looks like a masked SSN: `'***-**-****'`

In [0]:
%sql
-- =============================================================================
-- EXERCISE 3: Create the SSN masking function
-- This function checks group membership to determine masking level
--
-- Replace each <FILL_IN> with the correct value to make this cell run.
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- 1. SSN Masking: Full SSN for authorized, last 4 for research, fully masked for others
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER(<FILL_IN>) THEN ssn                                 -- Which group should see the FULL SSN? (the group name as a string)
    WHEN IS_MEMBER(<FILL_IN>) THEN CONCAT('***-**-', RIGHT(ssn, 4))    -- Which group gets only the last 4 digits?
    ELSE <FILL_IN>                                                      -- What should everyone else see? (fully masked SSN format)
  END;

### Step 3.2 - Create the Name Masking Functions

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** The cell completes successfully with no error
3. **What it does:** Creates two functions (`mask_first_name` and `mask_last_name`) that show:
   * Full names for `phi_authorized` and `billing_team`
   * Initials only (e.g., "M***") for `research_analysts`
   * `[REDACTED]` for everyone else

In [0]:
%sql
-- 2. Name Masking: Full name for authorized, initials for research, redacted for others
CREATE OR REPLACE FUNCTION mask_first_name(first_name STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN first_name
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(LEFT(first_name, 1), '***')
    WHEN IS_MEMBER('billing_team') THEN first_name
    ELSE '[REDACTED]'
  END;

CREATE OR REPLACE FUNCTION mask_last_name(last_name STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN last_name
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(LEFT(last_name, 1), '***')
    WHEN IS_MEMBER('billing_team') THEN last_name
    ELSE '[REDACTED]'
  END;

### Step 3.3 - Create Remaining Masking Functions (Date, Phone, Email, Address, Insurance)

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:**
   ```
   status
   ✅ All masking functions created successfully
   ```
3. **What it does:** Creates masking functions for 5 additional column types:
   * `mask_dob` — Date of birth (year only for researchers, redacted for others)
   * `mask_phone` — Phone number (visible to billing, hidden for others)
   * `mask_email` — Email (first 2 chars + domain for non-authorized users)
   * `mask_address` — Address (visible to billing and authorized, redacted for others)
   * `mask_insurance_id` — Insurance ID (last 3 digits only for non-authorized users)

In [0]:
%sql
-- 3. Date of Birth Masking: Full DOB for authorized, year only for research, redacted for others
CREATE OR REPLACE FUNCTION mask_dob(dob DATE)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN CAST(dob AS STRING)
    WHEN IS_MEMBER('research_analysts') THEN CONCAT(YEAR(dob), '-XX-XX')
    ELSE '[REDACTED]'
  END;

-- 4. Phone Number Masking
CREATE OR REPLACE FUNCTION mask_phone(phone STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN phone
    WHEN IS_MEMBER('billing_team') THEN phone
    ELSE '(XXX) XXX-XXXX'
  END;

-- 5. Email Masking
CREATE OR REPLACE FUNCTION mask_email(email STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN email
    ELSE CONCAT(LEFT(email, 2), '***@', SPLIT(email, '@')[1])
  END;

-- 6. Address Masking
CREATE OR REPLACE FUNCTION mask_address(addr STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN addr
    WHEN IS_MEMBER('billing_team') THEN addr
    ELSE '[REDACTED]'
  END;

-- 7. Insurance ID Masking
CREATE OR REPLACE FUNCTION mask_insurance_id(ins_id STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN ins_id
    WHEN IS_MEMBER('billing_team') THEN ins_id
    ELSE CONCAT('INS-*****', RIGHT(ins_id, 3))
  END;

SELECT '✅ All masking functions created successfully' AS status

### Step 3.4 - 🧪 Exercise 4: Apply the Masks to the Patients Table

> **This is the critical step.** Once you run this cell, ALL future queries against these columns will go through the masking functions.
> **This is a coding exercise.** Replace each `<FILL_IN>` with the correct masking function name.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct masking function name
2. Run the cell (**Shift + Enter**)
3. **What you should see:**
   ```
   status
   ✅ Column masks applied to 8 PHI columns on patients table
   ```
4. **What just happened:** Each of the 8 PHI columns (`ssn`, `first_name`, `last_name`, `date_of_birth`, `phone_number`, `email`, `address`, `insurance_id`) now has a masking function attached. Every query from this point forward will automatically run through the masks

**Hints:**
* All masking functions were created in Steps 3.1–3.3 and follow a naming pattern: `mask_<column_type>`
* For example: SSN → `mask_ssn`, first name → `mask_first_name`
* Date of birth uses an abbreviation: `mask_dob`

In [0]:
%sql
-- =============================================================================
-- EXERCISE 4: Apply the masking functions to the patients table columns
-- Once applied, ALL queries against these columns go through the mask
--
-- Replace each <FILL_IN> with the correct masking function name.
-- Hint: the function names follow the pattern mask_<column_type>
-- =============================================================================

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET MASK <FILL_IN>;                      -- Which function masks Social Security Numbers?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET MASK <FILL_IN>;               -- Which function masks first names?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET MASK <FILL_IN>;                -- Which function masks last names?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET MASK <FILL_IN>;            -- Which function masks dates of birth?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET MASK <FILL_IN>;             -- Which function masks phone numbers?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET MASK <FILL_IN>;                    -- Which function masks email addresses?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET MASK <FILL_IN>;                  -- Which function masks street addresses?

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET MASK <FILL_IN>;             -- Which function masks insurance IDs?

SELECT '✅ Column masks applied to 8 PHI columns on patients table' AS status

### Step 3.5 - See the Masking in Action!

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A table showing 10 patient records, but now the PHI columns are **masked**. Since you are likely not in any of the authorized groups on this trial workspace, you should see the most restrictive masking:
   * `first_name` → `[REDACTED]`
   * `last_name` → `[REDACTED]`
   * `ssn` → `***-**-****`
   * `date_of_birth` → `[REDACTED]`
   * `phone_number` → `(XXX) XXX-XXXX`
   * `email` → `ma***@gmail.com` (first 2 chars + domain)
   * `address` → `[REDACTED]`
   * `insurance_id` → `INS-*****789` (last 3 only)
3. **Compare this with Step 1.6!** The same SQL query now returns completely different results because the masking functions are active. No code changes needed — Unity Catalog handles it transparently
4. **Non-PHI columns** (`gender`, `race`, `insurance_type`, `blood_type`, `region`) remain **unmasked** and safe for analytics
5. Add yourself to the `research_analysts` group, then come back and re-run the masking demo cell to see partial masking in action!

In [0]:
%sql
-- =============================================================================
-- NOW query the same patients table - PHI columns are automatically masked!
-- Compare this with the "Before Masking" query above (Cell 5)
-- 
-- Since we're likely NOT in any of the authorized groups on this trial,
-- you should see the most restrictive masking applied.
-- =============================================================================

SELECT 
  patient_id,
  first_name    AS first_name_masked,
  last_name     AS last_name_masked,
  ssn           AS ssn_masked,
  date_of_birth AS dob_masked,
  phone_number  AS phone_masked,
  email         AS email_masked,
  address       AS address_masked,
  insurance_id  AS insurance_id_masked,
  -- These columns are NOT masked (analytics-safe)
  gender,
  race,
  insurance_type,
  blood_type,
  region,
  primary_department
FROM hipaa_demo.clinical_data.patients
LIMIT 10

> ✅ **Checkpoint:** Dynamic data masking is fully configured. Eight PHI columns are now automatically masked based on the querying user’s group membership. **Scroll down to continue to Section 4.**

---
# Section 4: Row-Level Security (Row Filters)

**Goal:** Restrict which **rows** a user can see based on their group membership. While Section 3 controlled which **columns** are visible, this section controls which **rows** are visible. Together, they implement the HIPAA **Minimum Necessary** principle.

---

### Background

**How do row filters work?** When you query a table that has a row filter, Unity Catalog evaluates a function for each row. The function returns `TRUE` (show the row) or `FALSE` (hide the row) based on who is running the query.

```
1. You run: SELECT * FROM encounters
2. Unity Catalog checks: Does this table have a row filter?
3. For EACH row, the filter function evaluates your group membership
4. You only see rows where the function returned TRUE
```

**Healthcare use cases:**
| Scenario | Filter Logic |
|----------|--------------|
| Regional access | Cardiologist in Northeast only sees Northeast patients |
| Department access | Oncology staff only sees oncology encounters |
| Research cohorts | Researchers only see patients who consented to research |
| Multi-tenant | Hospital A only sees Hospital A data in a shared platform |

Follow the steps below in order.

---

### Step 4.1 - 🧪 Exercise 5: Create the Region-Based Row Filter

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A confirmation that the function was created (no error)
4. **What it does:** Creates a function called `encounters_region_filter` that:
   * Returns ALL rows if you are in `phi_authorized` or `compliance_officers`
   * Returns only your region’s rows if you are in a regional team (e.g., `northeast_team` sees only `'Northeast'`)
   * Returns NO rows if you are not in any recognized group

**Hints:**
* Think about who needs to see everything for auditing purposes — that’s the group with "compliance" in its name
* The region string values match the group name pattern: `northeast_team` → `'Northeast'`, `midwest_team` → `'Midwest'`
* The default case (ELSE) should deny access — what boolean value means "no"?

In [0]:
%sql
-- =============================================================================
-- EXERCISE 5: Create a row filter function for the encounters table
-- Restricts which rows users can see based on their region assignment
--
-- Replace each <FILL_IN> with the correct value to make this cell run.
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- Row filter for encounters: Filter by region based on group membership
CREATE OR REPLACE FUNCTION encounters_region_filter(region_val STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    -- Admins and compliance officers see everything
    WHEN IS_MEMBER('phi_authorized') THEN TRUE
    WHEN IS_MEMBER(<FILL_IN>) THEN TRUE                                          -- Which other group should see ALL rows? (Hint: they audit access)
    -- Regional teams only see their region
    WHEN IS_MEMBER('northeast_team') AND region_val = <FILL_IN> THEN TRUE        -- What region string should northeast_team see?
    WHEN IS_MEMBER('southeast_team') AND region_val = 'Southeast' THEN TRUE
    WHEN IS_MEMBER('midwest_team') AND region_val = <FILL_IN> THEN TRUE          -- What region string should midwest_team see?
    WHEN IS_MEMBER('southwest_team') AND region_val = 'Southwest' THEN TRUE
    WHEN IS_MEMBER('west_team') AND region_val = 'West' THEN TRUE
    ELSE <FILL_IN>                                                                -- Should unrecognized users see any rows? (TRUE or FALSE)
  END;

### Step 4.3 - Apply the Row Filter to the Encounters Table

> **Important:** A table can only have **one** row filter at a time. This cell applies the region-based filter.

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A confirmation that the row filter was applied to the `encounters` table

In [0]:
%sql
-- =============================================================================
-- Apply the row filter to the encounters table
-- Note: A table can only have ONE row filter at a time.
-- We'll use the region filter as the primary demonstration.
-- =============================================================================

-- Apply region-based row filter to encounters
ALTER TABLE hipaa_demo.clinical_data.encounters 
  SET ROW FILTER encounters_region_filter ON (region);

### Step 4.4 - See the Row Filter in Action

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A table showing row counts grouped by `region`. Since you are the table owner, you will likely see **all regions** and all rows
3. **Key columns:** `region`, `encounter_count`, `unique_patients`
4. **In production:** Non-owner users would only see their assigned region’s data — other regions would be completely invisible to them

In [0]:
%sql
-- =============================================================================
-- Query encounters with the row filter active
-- As the table owner, you'll see all rows.
-- In production, non-owner users would only see their region's data.
-- =============================================================================

-- Show row counts by region (table owner sees all)
SELECT 
  region,
  COUNT(*) AS encounter_count,
  COUNT(DISTINCT patient_id) AS unique_patients,
  ROUND(AVG(total_charges), 2) AS avg_charges
FROM hipaa_demo.clinical_data.encounters
GROUP BY region
ORDER BY region

### Step 4.5 — Create Secure Analytics Views

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:**
   ```
   ✅ Three analytics views created
   ```
3. **What it does:** Creates a pre-built view that combine masking + row filters:
   * `v_top_diagnoses` — Diagnosis counts by region (no patient-level data)
4. **These views are how data scientists safely work with clinical data** — they never see PHI

In [0]:
%sql
-- =============================================================================
-- Create pre-built analytics view that combine masking + row filters
-- These provide safe, de-identified datasets for different user groups
-- =============================================================================

-- View: Top diagnoses by region (de-identified)
CREATE OR REPLACE VIEW hipaa_demo.clinical_data.v_top_diagnoses AS
SELECT
  e.region,
  d.icd10_code,
  d.diagnosis_description,
  COUNT(*) AS diagnosis_count,
  COUNT(DISTINCT d.patient_id) AS affected_patients,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY e.region), 2) AS pct_of_region
FROM hipaa_demo.clinical_data.diagnoses d
JOIN hipaa_demo.clinical_data.encounters e ON d.encounter_id = e.encounter_id
GROUP BY e.region, d.icd10_code, d.diagnosis_description;

SELECT '✅ Three analytics views created' AS status

### Step 4.6 - Run a Safe Analytics Query (No PHI Exposed)

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A results table showing the top 10 diagnoses with aggregate counts — with **no patient-identifiable information**
3. **This is exactly how a data scientist would work** with clinical data — they query a view that automatically strips PHI, so they never see sensitive information

In [0]:
%sql
-- =============================================================================
-- Example: Safe analytics query using de-identified view
-- Data scientists can run this without any PHI exposure!
-- =============================================================================

-- Top 10 diagnoses across all regions
SELECT 
  icd10_code,
  diagnosis_description,
  SUM(diagnosis_count) AS total_cases,
  SUM(affected_patients) AS total_patients_affected
FROM hipaa_demo.clinical_data.v_top_diagnoses
GROUP BY icd10_code, diagnosis_description
ORDER BY total_cases DESC
LIMIT 10

> ✅ **Checkpoint:** Row-level security is configured. You have also created secure analytics views that combine column masking and row filters for safe data access. **Scroll down to continue to Section 5.**

---
# Section 5: Audit Logging & HIPAA Compliance Monitoring

**Goal:** Query the Databricks system audit logs to see who accessed clinical data, how often, and whether any permission changes occurred. This is required by HIPAA §164.312(b).

---

### Background

**HIPAA audit requirements (§164.312(b))** say covered entities must:
1. **Record** all access to systems containing ePHI
2. **Review** activity logs regularly
3. **Investigate** anomalous access patterns
4. **Retain** audit logs for at least 6 years

**Databricks system tables** capture comprehensive audit data automatically:

| System Table | What It Captures |
|---|---|
| `system.access.audit` | All API calls, data access, authentication events |
| `system.information_schema.*` | Metadata about catalogs, schemas, tables, columns |
| `system.billing.usage` | Compute usage (useful for cost allocation per department) |

Follow the steps below in order.

---

### Step 5.1 - Check System Table Availability

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A result showing whether the `system.access.audit` table is available:
   ```
   table_name             | status
   system.access.audit    | ✅ Available
   ```
3. **If it shows an error or empty result:** That is OK for a trial workspace. Continue running the remaining cells anyway — the SQL is still useful to see what production audit queries look like

In [0]:
%sql
-- =============================================================================
-- Check if system audit tables are available on this workspace
-- =============================================================================

-- Try to access the audit log (may not be enabled on all trial workspaces)
SELECT 
  'system.access.audit' AS table_name,
  CASE 
    WHEN COUNT(*) >= 0 THEN '✅ Available'
    ELSE '❌ Not available'
  END AS status
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 1 DAY
LIMIT 1

### Step 5.2 - 🧪 Exercise 6: Query Recent Data Access Events

> **This is a coding exercise.** Replace each `<FILL_IN>` placeholder with the correct value.

1. Read through the code in the cell below and replace each `<FILL_IN>` with the correct value
2. Run the cell (**Shift + Enter**)
3. **What you should see:** A table showing recent data access events for your HIPAA tables, including:
   * `event_date` and `event_time` — when the access happened
   * `user_email` — who accessed the data
   * `action_name` — what they did (e.g., `commandSubmit`)
   * `table_accessed` — which table they queried
4. **If the query returns no rows:** This means no audit events have been recorded yet for the `hipaa_demo` tables. This is normal on a freshly set up workspace

**Hints:**
* The audit log lives in a system table with the path `system.access.audit`
* The request parameter that contains the table name is called `full_name_arg`
* For HIPAA compliance, 7 days is a reasonable minimum lookback window: `INTERVAL 7 DAY`

In [0]:
%sql
-- =============================================================================
-- EXERCISE 6: Who accessed our clinical data tables recently?
-- This query shows all data access events for our HIPAA-covered tables
--
-- Replace each <FILL_IN> with the correct value to make this cell run.
-- =============================================================================

SELECT 
  event_date,
  event_time,
  user_identity.email AS user_email,
  action_name,
  request_params.<FILL_IN> AS table_accessed,               -- Which request parameter field contains the fully qualified table name?
  source_ip_address,
  user_agent,
  response.status_code AS status
FROM <FILL_IN>                                              -- Which system table contains the audit logs? (catalog.schema.table)
WHERE event_date >= CURRENT_DATE() - INTERVAL <FILL_IN>     -- How far back should we look? (Hint: at least 7 days for HIPAA)
  AND (
    request_params.full_name_arg LIKE 'hipaa_demo.clinical_data.%'
    OR request_params.full_name_arg LIKE 'hipaa_demo.%'
  )
ORDER BY event_time DESC
LIMIT 50

### Step 5.3 - View Access Frequency by User and Table

1. Run the cell directly below (**Shift + Enter**)
2. **What you should see:** A summary table showing how many times each user accessed each table
3. **Why this matters:** This helps identify unusual access patterns — for example, a user querying the patients table 100 times in one day would stand out and trigger an investigation

In [0]:
%sql
-- =============================================================================
-- Query 2: Access frequency heatmap - which users access which tables most?
-- Critical for identifying unusual access patterns (HIPAA requirement)
-- =============================================================================

SELECT 
  user_identity.email AS user_email,
  request_params.full_name_arg AS table_name,
  COUNT(*) AS access_count,
  MIN(event_time) AS first_access,
  MAX(event_time) AS last_access,
  COUNT(DISTINCT event_date) AS days_accessed
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 30 DAY
  AND request_params.full_name_arg LIKE 'hipaa_demo.clinical_data.%'
  AND action_name IN ('commandSubmit', 'getTable', 'generateTemporaryTableCredential')
GROUP BY user_identity.email, request_params.full_name_arg
ORDER BY access_count DESC
LIMIT 25

> ✅ **Checkpoint:** You have queried the Databricks system audit logs and created a compliance monitoring view. In production, these queries would be scheduled to run daily and feed into a compliance dashboard. **Scroll down to continue to Section 6.**

---
# Section 6: Cleanup — Remove All Demo Resources

**Goal:** Tear down everything created by the three lab notebooks so your workspace is restored to its original state.

### Instructions

1. Run the **SQL cleanup cell** first (Step 6.1) — removes all Unity Catalog objects
2. Run the **Python cleanup cell** second (Step 6.2) — removes all security groups
3. Both cells are idempotent — safe to re-run if anything fails partway through

---

In [0]:
%sql
-- =============================================================================
-- CLEANUP: Remove all Unity Catalog objects created by this lab
-- Order matters: masks/filters → views → functions → tables → schema → catalog
-- =============================================================================

-- 1. Remove column masks from patients table
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN ssn DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN first_name DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN last_name DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN date_of_birth DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN phone_number DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN email DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN address DROP MASK;
ALTER TABLE hipaa_demo.clinical_data.patients ALTER COLUMN insurance_id DROP MASK;

-- 2. Remove row filter from encounters table
ALTER TABLE hipaa_demo.clinical_data.encounters DROP ROW FILTER;

-- 3. Drop the analytics view
DROP VIEW IF EXISTS hipaa_demo.clinical_data.v_top_diagnoses;

-- 4. Drop masking functions
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_ssn;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_first_name;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_last_name;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_dob;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_phone;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_email;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_address;
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.mask_insurance_id;

-- 5. Drop row filter function
DROP FUNCTION IF EXISTS hipaa_demo.clinical_data.encounters_region_filter;

-- 6. Drop all clinical data tables
DROP TABLE IF EXISTS hipaa_demo.clinical_data.patients;
DROP TABLE IF EXISTS hipaa_demo.clinical_data.encounters;
DROP TABLE IF EXISTS hipaa_demo.clinical_data.diagnoses;
DROP TABLE IF EXISTS hipaa_demo.clinical_data.lab_results;
DROP TABLE IF EXISTS hipaa_demo.clinical_data.medications;

-- 7. Drop the schema
DROP SCHEMA IF EXISTS hipaa_demo.clinical_data CASCADE;

-- 8. Drop the catalog
DROP CATALOG IF EXISTS hipaa_demo CASCADE;

SELECT '✅ All Unity Catalog objects removed (catalog, schema, tables, views, functions, masks, filters)' AS status

In [0]:
# =============================================================================
# CLEANUP: Remove all 17 security groups created by the HIPAA demo
# Uses the Databricks SDK — safe to re-run (skips groups that don't exist)
# =============================================================================

from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# All groups created by the demo
demo_groups = [
    # Access tier groups
    "phi_authorized",
    "research_analysts",
    "billing_team",
    "compliance_officers",
    # Regional groups
    "northeast_team",
    "southeast_team",
    "midwest_team",
    "southwest_team",
    "west_team",
    # Department groups
    "dept_cardiology",
    "dept_oncology",
    "dept_neurology",
    "dept_emergency",
    "dept_surgery",
    "dept_pediatrics",
    "dept_orthopedics",
    "dept_internal_med",
]

# Build a lookup of existing groups
existing_groups = {g.display_name: g for g in w.groups.list()}

deleted = 0
skipped = 0
failed = 0

for group_name in demo_groups:
    if group_name in existing_groups:
        try:
            w.groups.delete(id=existing_groups[group_name].id)
            print(f"  ✅ Deleted '{group_name}'")
            deleted += 1
        except Exception as e:
            print(f"  ❌ Failed to delete '{group_name}': {e}")
            failed += 1
    else:
        print(f"  ⏭️  '{group_name}' not found (skipped)")
        skipped += 1

print(f"\n🧹 Cleanup complete: {deleted} deleted, {skipped} skipped, {failed} failed")

> ✅ **Cleanup Complete!** All resources created by the HIPAA demo have been removed:
> * Unity Catalog objects (catalog, schema, tables, views, functions, masks, row filters)
> * All 17 security groups (access tier, regional, department)
>
> Your workspace is now back to its original state.

---

# Answer Key

> **⚠️ Spoiler alert!** The complete solutions for all 6 exercises are below. Try to solve each exercise on your own first — you’ll learn much more that way. Use these answers only when you’re truly stuck.

## Exercise 1 — Apply PHI Classification Tags to Patient Table

Copy the complete cell below into the Exercise 1 cell:

```sql
-- =============================================================================
-- EXERCISE 1: Apply sensitivity tags to columns containing PHI
-- Tags enable automated governance and compliance reporting
-- =============================================================================

-- First, set table-level tags
ALTER TABLE hipaa_demo.clinical_data.patients 
  SET TAGS ('data_classification' = 'PHI',
            'hipaa_covered' = 'true',
            'retention_policy' = '7_years',
            'data_owner' = 'clinical_ops');

-- Tag individual PHI columns
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET TAGS ('phi_category' = 'name', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET TAGS ('phi_category' = 'name', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET TAGS ('phi_category' = 'ssn', 'sensitivity' = 'critical');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET TAGS ('phi_category' = 'date', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET TAGS ('phi_category' = 'geographic', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET TAGS ('phi_category' = 'phone', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET TAGS ('phi_category' = 'email', 'sensitivity' = 'high');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET TAGS ('phi_category' = 'health_plan_id', 'sensitivity' = 'high');

-- Tag non-PHI columns as safe for analytics
ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN gender SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN race SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN blood_type SET TAGS ('sensitivity' = 'low', 'analytics_safe' = 'true');

SELECT '✅ Tags applied successfully to patients table' AS status
```

**Why these values?**
* `'PHI'` — The table contains Protected Health Information as defined by HIPAA. This is the standard classification label for any table with patient-identifiable data
* `'high'` for names — Patient names are one of the 18 HIPAA identifiers. They're "high" (not "critical") because names alone are less uniquely identifying than an SSN
* `'ssn'` — The PHI category for Social Security Numbers. SSN is the standard abbreviation used in healthcare data governance
* `'geographic'` — Addresses are geographic identifiers under HIPAA (any geographic subdivision smaller than a state)
* `'true'` — Gender is a demographic attribute that is safe for aggregate analytics without risking patient re-identification

## Exercise 2 — Query Tag Catalog for Compliance Report

Copy the complete cell below into the Exercise 2 cell:

```sql
-- =============================================================================
-- EXERCISE 2: Generate a compliance report — Which columns contain PHI?
-- This query uses the information_schema to find all tagged columns
-- =============================================================================

SELECT 
  t.table_name,
  c.column_name,
  c.data_type,
  ct.tag_name,
  ct.tag_value
FROM hipaa_demo.information_schema.tables t
JOIN hipaa_demo.information_schema.columns c 
  ON t.table_schema = c.table_schema AND t.table_name = c.table_name
LEFT JOIN hipaa_demo.information_schema.column_tags ct
  ON c.table_schema = ct.schema_name 
  AND c.table_name = ct.table_name 
  AND c.column_name = ct.column_name
WHERE t.table_schema = 'clinical_data'
  AND ct.tag_name IS NOT NULL
ORDER BY t.table_name, c.ordinal_position, ct.tag_name
```

**Why these values?**
* `c.table_name` — The JOIN between `tables` and `columns` needs to match on both `table_schema` (already provided) and `table_name`. This ensures each column is associated with the correct table
* `ct.column_name` — The LEFT JOIN to `column_tags` needs to match on `column_name` to link each tag back to the specific column it was applied to
* `'clinical_data'` — This is the schema name where all five clinical tables live (as shown in the three-level namespace diagram in Section 1)

## Exercise 3 — Create the SSN Masking Function

Copy the complete cell below into the Exercise 3 cell:

```sql
-- =============================================================================
-- EXERCISE 3: Create the SSN masking function
-- This function checks group membership to determine masking level
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- 1. SSN Masking: Full SSN for authorized, last 4 for research, fully masked for others
CREATE OR REPLACE FUNCTION mask_ssn(ssn STRING)
RETURNS STRING
RETURN 
  CASE 
    WHEN IS_MEMBER('phi_authorized') THEN ssn
    WHEN IS_MEMBER('research_analysts') THEN CONCAT('***-**-', RIGHT(ssn, 4))
    ELSE '***-**-****'
  END;
```

**Why these values?**
* `'phi_authorized'` — This is the group with full PHI access (see the access model table in Section 3). Clinical staff and treating physicians belong to this group and need to see the complete SSN for patient identification
* `'research_analysts'` — Researchers need some identifying information for record linkage but don't need the full SSN. Showing only the last 4 digits follows the "minimum necessary" principle (§164.502(b))
* `'***-**-****'` — For everyone else (data scientists, general analysts), the SSN is fully masked. The format preserves the visual structure of an SSN without revealing any digits

## Exercise 4 — Apply Column Masks to Patients Table

Copy the complete cell below into the Exercise 4 cell:

```sql
-- =============================================================================
-- EXERCISE 4: Apply the masking functions to the patients table columns
-- Once applied, ALL queries against these columns go through the mask
-- =============================================================================

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN ssn SET MASK mask_ssn;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN first_name SET MASK mask_first_name;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN last_name SET MASK mask_last_name;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN date_of_birth SET MASK mask_dob;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN phone_number SET MASK mask_phone;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN email SET MASK mask_email;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN address SET MASK mask_address;

ALTER TABLE hipaa_demo.clinical_data.patients 
  ALTER COLUMN insurance_id SET MASK mask_insurance_id;

SELECT '✅ Column masks applied to 8 PHI columns on patients table' AS status
```

**Why these values?**
* All masking functions follow a consistent naming convention: `mask_<column_type>`. This makes them easy to remember and maintain
* `mask_ssn` → SSN, `mask_first_name` → first name, `mask_last_name` → last name
* `mask_dob` → date of birth (abbreviated), `mask_phone` → phone number, `mask_email` → email
* `mask_address` → street address, `mask_insurance_id` → insurance ID
* The `ALTER TABLE ... ALTER COLUMN ... SET MASK` syntax is how Unity Catalog attaches a masking function to a column. Once set, the mask is enforced on every query automatically — no changes needed to existing queries or applications

## Exercise 5 — Create the Region-Based Row Filter

Copy the complete cell below into the Exercise 5 cell:

```sql
-- =============================================================================
-- EXERCISE 5: Create a row filter function for the encounters table
-- Restricts which rows users can see based on their region assignment
-- =============================================================================

USE CATALOG hipaa_demo;
USE SCHEMA clinical_data;

-- Row filter for encounters: Filter by region based on group membership
CREATE OR REPLACE FUNCTION encounters_region_filter(region_val STRING)
RETURNS BOOLEAN
RETURN 
  CASE
    -- Admins and compliance officers see everything
    WHEN IS_MEMBER('phi_authorized') THEN TRUE
    WHEN IS_MEMBER('compliance_officers') THEN TRUE
    -- Regional teams only see their region
    WHEN IS_MEMBER('northeast_team') AND region_val = 'Northeast' THEN TRUE
    WHEN IS_MEMBER('southeast_team') AND region_val = 'Southeast' THEN TRUE
    WHEN IS_MEMBER('midwest_team') AND region_val = 'Midwest' THEN TRUE
    WHEN IS_MEMBER('southwest_team') AND region_val = 'Southwest' THEN TRUE
    WHEN IS_MEMBER('west_team') AND region_val = 'West' THEN TRUE
    ELSE FALSE
  END;
```

**Why these values?**
* `'compliance_officers'` — Compliance officers need unrestricted access to all rows for auditing purposes. Along with `phi_authorized`, they are the only group that sees everything. This supports HIPAA audit requirements (§164.312(b))
* `'Northeast'` — The region string must match exactly what's stored in the `region` column. The naming convention is consistent: `northeast_team` → `'Northeast'`
* `'Midwest'` — Same pattern: `midwest_team` → `'Midwest'`. The region values use title case
* `FALSE` — The default case must deny access. If a user isn't in any recognized group, they should see zero rows. This implements the "deny by default" security principle that HIPAA requires

## Exercise 6 — Query Recent Data Access Events (Audit Logs)

Copy the complete cell below into the Exercise 6 cell:

```sql
-- =============================================================================
-- EXERCISE 6: Who accessed our clinical data tables recently?
-- This query shows all data access events for our HIPAA-covered tables
-- =============================================================================

SELECT 
  event_date,
  event_time,
  user_identity.email AS user_email,
  action_name,
  request_params.full_name_arg AS table_accessed,
  source_ip_address,
  user_agent,
  response.status_code AS status
FROM system.access.audit
WHERE event_date >= CURRENT_DATE() - INTERVAL 7 DAY
  AND (
    request_params.full_name_arg LIKE 'hipaa_demo.clinical_data.%'
    OR request_params.full_name_arg LIKE 'hipaa_demo.%'
  )
ORDER BY event_time DESC
LIMIT 50
```

**Why these values?**
* `full_name_arg` — This is the field within `request_params` that contains the fully qualified table name (e.g., `hipaa_demo.clinical_data.patients`). It's the standard parameter Databricks uses to log which table was accessed
* `system.access.audit` — This is the Databricks system table that captures all audit events. It's the foundation of HIPAA compliance monitoring (§164.312(b)). The three-level path follows Unity Catalog naming: catalog (`system`) → schema (`access`) → table (`audit`)
* `7 DAY` — HIPAA requires regular review of access logs. A 7-day lookback window is a practical starting point for routine compliance checks. In production, you would also run broader 30-day and 90-day reviews